# Column Schema and the Battery Data Format
This notebook describes how columns are internally referenced inside of PyProBE.

As demonstrated by other examples, when we have data stored inside a PyProBE `Table` or `CyclingData` object, we are able to return the columns with any unit:

In [ ]:
from pprint import pprint

import pyprobe

data_directory = "../../../tests/sample_data/neware"

# Create a cell object
procedure = pyprobe.Procedure.load(
    data_directory + "/sample_data_neware.bdf.parquet",
)
print("Current / A:", procedure.get("Current / A")[0:5])
print("Current / mA:", procedure.get("Current / mA")[0:5])

The `Procedure` (and all `Table` objects) have a `columns` property, which is an instance of the `ColumnDict` class, a mapping between the column name and a python object that represents it.

In [ ]:
print(procedure.columns)

This class has two other static attributes:
- `ColumnSet.names` returns a tuple of the column name strings
- `ColumnSet.quantities` reutrns a tuple of the quantities of each column

In [ ]:
print("Names: ", procedure.columns.names)
print("Quantities: ", procedure.columns.quantities)

The `values` of the mapping are instances of the `Column` class, which you can retrieve by indexing the `ColumnDict`:

In [ ]:
print(repr(procedure.columns["Unix Time / s"]))

It is on this class that conversion is applied. This is done with the `resolve()` method. This returns the Polars expression for a particular column. Resolving a column that already exists, just returns the column:

In [ ]:
print(procedure.columns.resolve("Current / A"))

This allows unit conversions:

In [ ]:
print(procedure.columns.resolve("Current / mA"))

PyProBE uses the [Battery Data Format](https://github.com/battery-data-alliance/battery-data-format) for its column schema. This provides a set of uniquely defined quantities that can be used across the code. Since they have static definitions, we can define relationships between them. This allows certain BDF columns to be calculated whether or not they are in the data. These 'recipes' are stored in a persistent attribute `pyprobe.columns.BDF_RECIPES`

In [ ]:
pprint(pyprobe.columns.BDF_RECIPES)

We are going to use the example of `Test Time / s`, which can be derived from `Unix Time / s` by simply subtracting the first value. We'll first drop the column from the data:

In [ ]:
procedure.lf = procedure.lf.drop("Test Time / s")
print(procedure.columns)

Then show that we can retrieve it anyway:

In [ ]:
print(procedure.get("Test Time / s"))

In PyProBE, the BDF columns are stored persistently in the `pyprobe.columns.BDF` attribute:

In [ ]:
from pyprobe.columns import BDF

print("Column Name:", BDF.CURRENT_AMPERE.name, "\n", repr(BDF.CURRENT_AMPERE))
for bdf_col in BDF:
    print("Column Name:", bdf_col.name, "\n", repr(bdf_col))

You can use them anywhere in place of the string column names. You cannot call `BDF_VOLTAGE_MILLIVOLT` for unit conversion, but for calculations in SI units, these attributes are more python-native and less error-prone. All internal calculations in PyProBE are done this way.

In [ ]:
print(procedure.get(BDF.VOLTAGE_VOLT))

In [ ]:
print(procedure.columns.resolve(BDF.NET_CAPACITY_AH))